# Shallow and Deep Networks


Figures are from Prince, *[Understanding Deep Learning](https://udlbook.github.io/udlbook/)*
(UDL, chapters 3 and 4), unless another source is cited.
Code cells adapted from the UDL notebooks (MIT license,
[github.com/udlbook/udlbook/tree/main/Notebooks](https://github.com/udlbook/udlbook/tree/main/Notebooks)).


The 1D linear regression model of last lecture is obviously limited: we want
input/output mappings that are not lines, and we want multiple inputs and
multiple outputs. A **shallow neural network** (one hidden layer) is flexible
enough to describe arbitrarily complex mappings with as many inputs and
outputs as we want. A **deep neural network** (more than one hidden layer)
composes shallow networks, and gets the same flexibility with far fewer
parameters.


## This lecture we'll cover…

**Shallow networks**

- Example network, 1 input, 1 output
- Universal approximation theorem
- More than one output, more than one input
- General case and number of regions
- Terminology


**Deep networks**

- Composing two networks
- Combining them into one
- Hyperparameters: depth and width
- Matrix notation and the general case
- Shallow vs. deep networks


You have met multi-layer perceptrons before (DS701 or DS340), so the first part
moves quickly through what a hidden unit is. The ideas to slow down on are the
ones a quiz can ask about: why a ReLU network is piecewise linear, what the
universal approximation theorem does and does not promise, how to count linear
regions and parameters, and why depth buys more regions per parameter than
width.


# Example network, 1 input, 1 output

We start from the 1D linear regression model and build the simplest shallow
network: one input, one output and three hidden units.


## Example shallow network

**1D linear regression:**

$$
y = \mathrm{f}[x, \boldsymbol{\phi}] = \phi_0 + \phi_1 x
$$

**Example shallow network:**

$$
\begin{aligned}
y &= \mathrm{f}[x, \boldsymbol{\phi}] \\
&= \phi_0 + \phi_1 \mathrm{a}[\theta_{10} + \theta_{11} x] + \phi_2 \mathrm{a}[\theta_{20} + \theta_{21} x] + \phi_3 \mathrm{a}[\theta_{30} + \theta_{31} x]
\end{aligned}
$$


This model has 10 parameters:

$$
\boldsymbol{\phi} = \{\phi_0, \phi_1, \phi_2, \phi_3, \theta_{10}, \theta_{11}, \theta_{20}, \theta_{21}, \theta_{30}, \theta_{31}\}
$$

- Represents a family of functions; the parameters pick one function
- Given the parameters, inference is evaluating the equation
- Given training data $\{x_i, y_i\}_{i=1}^{I}$, define a loss $L[\boldsymbol{\phi}]$
  (least squares) and change the parameters to minimize it


## The activation function: ReLU

$$
y = \phi_0 + \phi_1 \mathrm{a}[\theta_{10} + \theta_{11} x] + \phi_2 \mathrm{a}[\theta_{20} + \theta_{21} x] + \phi_3 \mathrm{a}[\theta_{30} + \theta_{31} x]
$$

$\mathrm{a}[\cdot]$ is the **activation function**.

$$
\mathrm{a}[z] = \mathrm{ReLU}[z] = \begin{cases} 0 & \quad z < 0 \\ z & \quad z \geq 0 \end{cases}.
$$

**Rectified Linear Unit** (a particular kind of activation function)


![](figs/02-Shallow-and-Deep-Networks/relu-active-inactive-regions.png)


## Hidden units

$$
y = \phi_0 + \phi_1 \mathrm{a}[\theta_{10} + \theta_{11} x] + \phi_2 \mathrm{a}[\theta_{20} + \theta_{21} x] + \phi_3 \mathrm{a}[\theta_{30} + \theta_{31} x].
$$

Break down into two parts:

$$
y = \phi_0 + \phi_1 h_1 + \phi_2 h_2 + \phi_3 h_3
$$

where the **hidden units** are:

$$
\begin{aligned}
h_1 &= \mathrm{a}[\theta_{10} + \theta_{11} x] \\
h_2 &= \mathrm{a}[\theta_{20} + \theta_{21} x] \\
h_3 &= \mathrm{a}[\theta_{30} + \theta_{31} x]
\end{aligned}
$$

## Building the output in four steps

1. Compute three linear functions of $x$
2. Pass each through the ReLU to get the hidden units $h_1, h_2, h_3$
3. Weight the hidden units by $\phi_1, \phi_2, \phi_3$
4. Sum them and add the offset $\phi_0$


![Steps 3 and 4: the weighted hidden units (top row) are summed, with the offset $\phi_0$, to give the output (bottom). The joints of the output, marked in red, line up with the joints of the hidden units.](figs/02-Shallow-and-Deep-Networks/step4-sum-weighted-hidden-units.png)

Here is the same construction in code. `theta` holds one row
$(\theta_{d0}, \theta_{d1})$ per hidden unit and `phi` holds
$(\phi_0, \phi_1, \phi_2, \phi_3)$. Change one entry and watch which joint
moves: $\theta_{d0}$ and $\theta_{d1}$ set where joint $d$ sits
($x = -\theta_{d0}/\theta_{d1}$), while $\phi_d$ only changes how sharply the
output bends there.

In [ ]:
#| code-fold: true
#| fig-cap: "The four steps for a 1-3-1 ReLU network. Rows 1 to 3: the lines, the hidden units and the weighted hidden units. Bottom: their sum plus $\\phi_0$, with the joints marked in red."
#| fig-alt: "A 3 by 3 grid of small plots over x from 0 to 2 showing, for each of three hidden units, its straight line, the line clipped at zero by the ReLU, and the clipped line scaled by phi. Below, a wide plot of the output, a piecewise linear function with three joints marked by red dotted lines at x = 0.6, 1.0 and 1.54."
# Adapted from UDL notebook 3.1 (MIT)
import numpy as np
import matplotlib.pyplot as plt

def relu(z):
    return np.clip(z, 0.0, None)

# Row d of theta is (theta_d0, theta_d1); phi is (phi_0, phi_1, phi_2, phi_3).
theta = np.array([[ 0.3, -0.5  ],
                  [-1.0,  1.0  ],
                  [-0.5,  0.325]])
phi = np.array([-0.3, 2.0, -1.0, 7.0])

def shallow_1_3_1(x, theta, phi, activation=relu):
    pre = theta[:, [0]] + theta[:, [1]] * x      # step 1: one line per hidden unit
    h = activation(pre)                           # step 2: hidden units
    weighted = phi[1:, None] * h                  # step 3: weight them
    y = phi[0] + weighted.sum(axis=0)             # step 4: sum and add the offset
    return pre, h, weighted, y

x = np.linspace(0, 2, 401)
pre, h, weighted, y = shallow_1_3_1(x, theta, phi)
joints = -theta[:, 0] / theta[:, 1]

fig = plt.figure(figsize=(8, 8))
grid = fig.add_gridspec(4, 3, hspace=0.6, wspace=0.35)
rows = [(pre, r"$\theta_{%d0} + \theta_{%d1} x$"), (h, "$h_%d$"), (weighted, r"$\phi_%d h_%d$")]
for r, (values, label) in enumerate(rows):
    for d in range(3):
        ax = fig.add_subplot(grid[r, d])
        ax.plot(x, values[d], color=f"C{d}")
        ax.axhline(0, color="0.8", lw=0.8)
        ax.set_xlim(0, 2)
        ax.set_title(label.replace("%d", str(d + 1)), fontsize=9)
ax = fig.add_subplot(grid[3, :])
ax.plot(x, y, "k")
for xj in joints[(joints >= 0) & (joints <= 2)]:
    ax.axvline(xj, color="r", ls=":", lw=1)
ax.set_xlim(0, 2)
ax.set_xlabel("Input, $x$")
ax.set_title(r"$y = \phi_0 + \phi_1 h_1 + \phi_2 h_2 + \phi_3 h_3$", fontsize=10)
plt.show()

print("joints at x =", np.round(np.sort(joints), 3))

## Try it: the four steps, interactively


[Interactive version of UDL Figure 3.3, ported from the UDL interactive figures (MIT).]{style="font-size: 0.7em;"}

Each column of sliders controls one hidden unit: its intercept
$\theta_{d0}$, slope $\theta_{d1}$ and output weight $\phi_d$. Changing the
activation resets the sliders to UDL's defaults for it. The NumPy cell under
*Building the output in four steps* computes the same panels, and the one under
*Other activation functions* compares the three activations.


## Shallow network = piecewise linear function

$$
y = \phi_0 + \phi_1 \mathrm{a}[\theta_{10} + \theta_{11} x] + \phi_2 \mathrm{a}[\theta_{20} + \theta_{21} x] + \phi_3 \mathrm{a}[\theta_{30} + \theta_{31} x].
$$

![Three different parameter settings of the same network. Each is a piecewise linear function with three joints.](figs/02-Shallow-and-Deep-Networks/shallow-network-example-functions.png)

1 "joint" per ReLU function, so 3 hidden units give (at most) 4 linear regions


## Activation pattern = which hidden units are active?

![](figs/02-Shallow-and-Deep-Networks/activation-pattern.png)


Shaded region:

- Unit 1 active
- Unit 2 inactive
- Unit 3 active

Within one region the activation pattern is fixed, so the output is linear
there.


## Depicting neural networks

$$
h_1 = \mathrm{a}[\theta_{10} + \theta_{11} x] \qquad
h_2 = \mathrm{a}[\theta_{20} + \theta_{21} x] \qquad
h_3 = \mathrm{a}[\theta_{30} + \theta_{31} x]
$$

$$
y = \phi_0 + \phi_1 h_1 + \phi_2 h_2 + \phi_3 h_3
$$

![Each parameter multiplies its source and adds to its target.](figs/02-Shallow-and-Deep-Networks/network-diagram-with-biases.png)


![Usually the bias terms are not drawn.](figs/02-Shallow-and-Deep-Networks/network-diagram-1-3-1.png)


# Universal approximation theorem

Nothing limits the network to three hidden units. Adding hidden units adds
joints, and with enough of them the network can approximate any 1D function.


## With enough hidden units…

**With $D$ hidden units** (instead of 3):

$$
h_d = \mathrm{a}[\theta_{d0} + \theta_{d1} x]
\qquad\qquad
y = \phi_0 + \sum_{d=1}^{D} \phi_d h_d
$$

$D$ hidden units give up to $D$ joints and $D + 1$ linear regions.

… so with enough of them we can describe any 1D function to arbitrary accuracy

![](figs/02-Shallow-and-Deep-Networks/universal-approximation-linear-regions.png)


**Universal approximation theorem:** with enough hidden units, there exists a
shallow neural network that can describe any given continuous function defined
on a compact subset of $\mathbb{R}^{D_i}$ to arbitrary precision (UDL section 3.2)



"Up to $D + 1$" is easy to check numerically. The function below counts the
linear pieces of a sampled curve by looking for changes of slope, without
looking inside the network. A random network with $D$ hidden units has exactly
$D$ joints somewhere on the real line, so over a range that contains all of them
the count is $D + 1$. Over a fixed window such as $[-1, 1]$ many joints fall
outside and the count is smaller: "up to" matters.

In [ ]:
def count_linear_regions(x, y, tol=1e-6):
    """Count the linear pieces of a sampled piecewise linear function from its slope changes."""
    slope = np.diff(y) / np.diff(x)
    changed = np.abs(np.diff(slope)) > tol
    # A joint inside a grid cell changes the slope on both sides of that cell,
    # so count each run of consecutive changes once.
    starts = changed & ~np.r_[False, changed[:-1]]
    return 1 + int(starts.sum())

def network(x, theta, phi):
    """Output of a shallow network with any number of hidden units, one unit at a time."""
    y = np.full_like(x, phi[0])
    for (t0, t1), p in zip(theta, phi[1:]):
        y += p * relu(t0 + t1 * x)
    return y

rng = np.random.default_rng(0)
print(f"{'D':>4} {'D + 1':>6} {'all joints':>11} {'on [-1, 1]':>11}")
for D in [3, 10, 50]:
    theta_r = rng.normal(size=(D, 2))
    phi_r = rng.normal(size=D + 1)
    joints_r = -theta_r[:, 0] / theta_r[:, 1]
    x_all = np.linspace(joints_r.min() - 1, joints_r.max() + 1, 2_000_001)
    x_win = np.linspace(-1, 1, 200_001)
    n_all = count_linear_regions(x_all, network(x_all, theta_r, phi_r))
    n_win = count_linear_regions(x_win, network(x_win, theta_r, phi_r))
    print(f"{D:>4} {D + 1:>6} {n_all:>11} {n_win:>11}")

# More than one output

The same hidden units can feed more than one output. Each output is its own
weighted sum of the hidden units.


## Two outputs

- 1 input, 4 hidden units, 2 outputs

$$
\begin{aligned}
h_1 &= \mathrm{a}[\theta_{10} + \theta_{11} x] \\
h_2 &= \mathrm{a}[\theta_{20} + \theta_{21} x] \\
h_3 &= \mathrm{a}[\theta_{30} + \theta_{31} x] \\
h_4 &= \mathrm{a}[\theta_{40} + \theta_{41} x]
\end{aligned}
\qquad\qquad
\begin{aligned}
y_1 &= \phi_{10} + \phi_{11} h_1 + \phi_{12} h_2 + \phi_{13} h_3 + \phi_{14} h_4 \\
y_2 &= \phi_{20} + \phi_{21} h_1 + \phi_{22} h_2 + \phi_{23} h_3 + \phi_{24} h_4
\end{aligned}
$$

![](figs/02-Shallow-and-Deep-Networks/network-diagram-1-4-2.png)


![](figs/02-Shallow-and-Deep-Networks/two-outputs-functions.png)



# More than one input

With two inputs, each hidden unit starts from a linear function of both inputs:
a plane over the $(x_1, x_2)$ input space instead of a line. The ReLU clips
that plane to zero on one side of a straight boundary, and summing the clipped
planes cuts the input space into convex polygonal regions.


## Two inputs

- 2 inputs, 3 hidden units, 1 output

$$
\begin{aligned}
h_1 &= \mathrm{a}[\theta_{10} + \theta_{11} x_1 + \theta_{12} x_2] \\
h_2 &= \mathrm{a}[\theta_{20} + \theta_{21} x_1 + \theta_{22} x_2] \\
h_3 &= \mathrm{a}[\theta_{30} + \theta_{31} x_1 + \theta_{32} x_2]
\end{aligned}
\qquad\qquad
y = \phi_0 + \phi_1 h_1 + \phi_2 h_2 + \phi_3 h_3
$$

![](figs/02-Shallow-and-Deep-Networks/network-diagram-2-3-1.png)

## Two inputs: linear functions and activation

A linear function of two inputs is a plane; color shows its value.


[Interactive version of UDL Figure 3.8a, ported from the UDL interactive figures (MIT).]{style="font-size: 0.7em;"}


![Linear functions.](figs/02-Shallow-and-Deep-Networks/two-inputs-linear-functions.png)

![After activation.](figs/02-Shallow-and-Deep-Networks/two-inputs-after-relu.png)


## Two inputs: weight and sum the hidden units

![Weight the hidden units.](figs/02-Shallow-and-Deep-Networks/two-inputs-weighted-hidden-units.png)


![Sum the weighted hidden units.](figs/02-Shallow-and-Deep-Networks/two-inputs-output-regions.png)


**Convex polygonal regions**

A region of $\mathbb{R}^{D_i}$ is convex if the straight line segment between
any two points in the region lies entirely inside the region.


## Try it: a two-input network


[Interactive version of UDL Figure 3.8b, ported from the UDL interactive figures (MIT).]{style="font-size: 0.7em;"}

Rows: the three pre-activations, the hidden units and the weighted hidden
units; the output is at the bottom right. The cyan lines mark where each
pre-activation is zero: the joints of the ReLU, now lines in the input plane.


## The output is a piecewise planar surface

![The same output as a 3D surface: one plane per region.](figs/02-Shallow-and-Deep-Networks/two-inputs-output-3d.png)


![Fitting a dataset where each sample has 2 inputs and 1 output.](figs/02-Shallow-and-Deep-Networks/two-inputs-fit-3d.png)


**Question:** if this network had a second output, what would it look like?



# General case

Putting the two extensions together gives a shallow network with any number of
inputs, hidden units and outputs.


## Arbitrary inputs, hidden units, outputs

- $D_i$ inputs, $D$ hidden units, and $D_o$ outputs

$$
h_d = \mathrm{a}\left[\theta_{d0} + \sum_{i=1}^{D_i} \theta_{di} x_i\right]
\qquad\qquad
y_j = \phi_{j0} + \sum_{d=1}^{D} \phi_{jd} h_d
$$

- e.g., three inputs, three hidden units, two outputs

![](figs/02-Shallow-and-Deep-Networks/network-diagram-3-3-2.png)

**Question:** how many parameters does this model have?



## How would you draw and write this neural network?

![](figs/02-Shallow-and-Deep-Networks/how-many-hidden-units.png)


![](figs/02-Shallow-and-Deep-Networks/neural-network-diagram-labeled.png)

$$
y = \phi_0 + \phi_1 h_1 + \phi_2 h_2 + \phi_3 h_3
$$

$$
\begin{aligned}
h_1 &= \mathrm{a}[\theta_{10} + \theta_{11} x_1 + \theta_{12} x_2] \\
h_2 &= \mathrm{a}[\theta_{20} + \theta_{21} x_1 + \theta_{22} x_2] \\
h_3 &= \mathrm{a}[\theta_{30} + \theta_{31} x_1 + \theta_{32} x_2]
\end{aligned}
$$



# Number of regions

Each hidden unit adds a boundary (a hyperplane) to the input space, so the
number of linear regions grows with the number of hidden units and with the
input dimension. With ReLU activations each output is a piecewise linear
function made of convex polytopes, and we can count them.


## Example with $D = D_i$ → $2^{D_i}$ regions

> $D_i$ : \# of inputs, $D$ : \# of hidden units, $D_o$ : \# of outputs

![](figs/02-Shallow-and-Deep-Networks/regions-with-D-equal-Di.png)

- 1 input (1-dimension)
- 1 hidden unit
- creates two regions (one joint)


- 2 inputs (2-dimensions) with
- 2 hidden units
- creates four regions (two lines)


- 3 inputs (3-dimensions) with
- 3 hidden units
- creates eight regions (three planes)


## Number of regions: $D > D_i$ hidden units

> $D_i$ : \# of inputs, $D$ : \# of hidden units, $D_o$ : \# of outputs

- The maximum number of regions created by $D > D_i$ hyperplanes in $D_i$
  dimensions was proved by Zaslavsky (1975) to be

$$
\sum_{j=0}^{D_i} \binom{D}{j},
\qquad \text{where } \binom{D}{j} = \frac{D!}{j!(D-j)!} \text{ are the binomial coefficients}
$$

(UDL section 3.4 and chapter 3 notes)

- How big is this? Greater than $2^{D_i}$ but less than $2^{D}$.
- Check: with two inputs and three hidden units,
  $\binom{3}{0} + \binom{3}{1} + \binom{3}{2} = 1 + 3 + 3 = 7$ regions, the
  seven polygons we saw.


## Number of regions grows very fast

- In general, each output consists of $D_i$-dimensional **convex polytopes**
- How many?

![Highlighted point = 500 hidden units or 51,001 parameters.](figs/02-Shallow-and-Deep-Networks/number-of-regions-plots.png)


Zaslavsky's formula as a calculator. The `min` handles $D < D_i$, where the
$D$ hyperplanes can cut at most $2^D$ regions. Check the seven regions of the
two-input example and the highlighted point of the figure above.

In [ ]:
# Adapted from UDL notebook 3.3 (MIT)
from math import comb, log10

def max_regions_shallow(D_i, D):
    """Zaslavsky (1975): most regions D hyperplanes can cut R^{D_i} into."""
    return sum(comb(D, j) for j in range(min(D_i, D) + 1))

def params_shallow(D_i, D, D_o=1):
    return D * (D_i + 1) + D_o * (D + 1)

for D_i, D in [(1, 3), (2, 3), (3, 3), (10, 50)]:
    print(f"D_i = {D_i:>3}, D = {D:>3}: {max_regions_shallow(D_i, D):,} regions")

D_i, D = 100, 500
N = max_regions_shallow(D_i, D)
print(f"D_i = {D_i}, D = {D}: about 10^{log10(N):.1f} regions, "
      f"{params_shallow(D_i, D):,} parameters")
print("2^D_i < N < 2^D:", 2**D_i < N < 2**D)

# Terminology

The names used for the parts of a network, and some alternatives to the ReLU
activation function.


## Nomenclature

![](figs/02-Shallow-and-Deep-Networks/nomenclature-network.png)


- Y-offsets = **biases**
- Slopes = **weights**
- Everything in one layer connected to everything in the next = **fully connected network (multi-layer perceptron)**
- No loops = **feedforward network**
- Values after ReLU (activation functions) = **activations**
- Values before ReLU = **pre-activations**
- One hidden layer = **shallow neural network**
- More than one hidden layer = **deep neural network**
- Number of hidden units $\approx$ **capacity**


## Other activation functions

![Figure 3.13: Prince, *Understanding Deep Learning*. Swish: Ramachandran, P., Zoph, B., & Le, Q. V. (2017). Searching for activation functions. [arXiv:1710.05941](https://arxiv.org/abs/1710.05941).](figs/02-Shallow-and-Deep-Networks/other-activation-functions.png)

Everything above about joints and regions is specific to the ReLU. A smooth
activation such as the sigmoid gives a smooth output instead of a piecewise
linear one, but the construction (linear function, activation, weight, sum) is
the same. Switch the interactive Figure 3.3 under *Try it: the four steps,
interactively* to the sigmoid or the Heaviside step to compare.


In [ ]:
#| code-fold: true
#| fig-cap: "The same 1-3-1 network (same $\\theta$, with $\\phi = (0.3, 0.5, -1.0, 0.9)$) with three activation functions."
#| fig-alt: "Three plots of output y against x from 0 to 2 for one network with different activations: with ReLU a piecewise linear curve with sharp joints; with a sigmoid a smooth curve with gentle bends at the same places; with a Heaviside step a staircase that jumps at the same x positions."
# Adapted from UDL notebook 3.4 (MIT)
def sigmoid(z):
    return 1 / (1 + np.exp(-10 * z))

def heaviside(z):
    return (z >= 0).astype(float)

phi_smooth = np.array([0.3, 0.5, -1.0, 0.9])
fig, axes = plt.subplots(1, 3, figsize=(9, 2.8), sharey=True)
for ax, (name, fn) in zip(axes, [("ReLU", relu), ("sig", sigmoid), ("heaviside", heaviside)]):
    ax.plot(x, shallow_1_3_1(x, theta, phi_smooth, activation=fn)[3], "k")
    ax.set_title(f"a[z] = {name}[z]", fontsize=10)
    ax.set_xlabel("Input, $x$")
axes[0].set_ylabel("Output, $y$")
plt.tight_layout()
plt.show()

# Composing two networks

A shallow network has a single hidden layer. We start building deep networks by
feeding the output of one shallow network into a second one, and we will find
that the composed function has many more linear regions than either network
alone.


## Composing two networks

**Network 1:**

$$
\begin{aligned}
h_1 &= \mathrm{a}[\theta_{10} + \theta_{11} x] \\
h_2 &= \mathrm{a}[\theta_{20} + \theta_{21} x] \\
h_3 &= \mathrm{a}[\theta_{30} + \theta_{31} x]
\end{aligned}
\qquad\qquad
y = \phi_0 + \phi_1 h_1 + \phi_2 h_2 + \phi_3 h_3
$$

**Network 2:**

$$
\begin{aligned}
h'_1 &= \mathrm{a}[\theta'_{10} + \theta'_{11} y] \\
h'_2 &= \mathrm{a}[\theta'_{20} + \theta'_{21} y] \\
h'_3 &= \mathrm{a}[\theta'_{30} + \theta'_{31} y]
\end{aligned}
\qquad\qquad
y' = \phi'_0 + \phi'_1 h'_1 + \phi'_2 h'_2 + \phi'_3 h'_3
$$

![](figs/02-Shallow-and-Deep-Networks/two-networks-composed-diagram.png)

## Composing two networks: example

Assume:

- ReLU activation
- Slopes and intercepts as shown
- 3 hidden units in each


Example: pick parameters so that $x \in [-1, 1]$ maps to $y \in [-1, 1]$ with
alternating slope

![](figs/02-Shallow-and-Deep-Networks/composing-example-two-functions.png)


**Let's see what happens when we map $x \rightarrow y \rightarrow y'$**


## Mapping $x \rightarrow y \rightarrow y'$


Sweep an input $x$ from $-1$ to $1$ and trace it through both networks. Panel
d) shows the result: the composed function $y'$ of $x$. Because three values of
$x$ map to the same $y$, the second network's function is replicated three
times (and flipped on alternate pieces).

![a) The composed network. b) The first network maps $x$ to $y$. c) The second network maps $y$ to $y'$. d) The composed function $y'$ of $x$.](figs/02-Shallow-and-Deep-Networks/composition-sweep-01.png)


The composition in code: run network 1, then feed its output into network 2.
The parameters are those of UDL notebook 4.1. Try flipping the sign of one
$\phi$ in network 2 and predict the composed curve before you run it.

In [ ]:
#| code-fold: true
#| fig-cap: "Network 1 maps $x$ to $y$, network 2 maps $y$ to $y'$, and their composition maps $x$ to $y'$."
#| fig-alt: "Three square plots on [-1, 1]. Left: network 1, a zigzag in x with three linear pieces. Middle: network 2 as a function of its input, with four linear pieces. Right: the composition, a zigzag in x with twelve linear pieces in which network 2's curve appears three times, flipped on the middle piece."
# Adapted from UDL notebook 4.1 (MIT)
theta_1 = np.array([[ 0.0, -1.0], [0.0, 1.0], [-0.67, 1.0]])
phi_1 = np.array([1.0, -2.0, -3.0, 9.3])
theta_2 = np.array([[-0.6, -1.0], [0.2, 1.0], [-0.5, 1.0]])
phi_2 = np.array([0.5, -1.0, -1.5, 2.0])

x = np.linspace(-1, 1, 200_001)
y1 = shallow_1_3_1(x, theta_1, phi_1)[3]        # network 1: x -> y
y2 = shallow_1_3_1(x, theta_2, phi_2)[3]        # network 2 on its own, input on [-1, 1]
y12 = shallow_1_3_1(y1, theta_2, phi_2)[3]      # composed: x -> y -> y'

fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
panels = [(y1, "Network 1", "$x$", "$y$"),
          (y2, "Network 2", "$y$", "$y'$"),
          (y12, "Composed", "$x$", "$y'$")]
for ax, (curve, title, xl, yl) in zip(axes, panels):
    ax.plot(x, curve, "k")
    ax.set(xlim=(-1, 1), ylim=(-1, 1), title=title, xlabel=xl, ylabel=yl, aspect="equal")
plt.tight_layout()
plt.show()

for name, curve in [("network 1", y1), ("network 2", y2), ("composed", y12)]:
    print(f"{name:>9}: {count_linear_regions(x, curve):>2} linear regions on [-1, 1]")

Network 1 sweeps $[-1, 1]$ three times, so each of network 2's four pieces
appears three times: $3 \times 4 = 12$ regions from two 10-parameter networks.


## Try it: compose two networks

Drag the handles to reshape each network; the right panel is network 2
applied to the output of network 1.




[Interactive version of UDL Figure 4.1, ported from the UDL interactive figures (MIT).]{style="font-size: 0.7em;"}

Each network is set by four points: the end points move up and down, the inner
two also left and right, and the network's three ReLU joints follow them. The
NumPy cell under *Mapping $x \rightarrow y \rightarrow y'$* composes the two
networks of UDL notebook 4.1 the same way.


## "Folding analogy"

![](figs/02-Shallow-and-Deep-Networks/folding-analogy.png)


## Comparing to a shallow network with six hidden units

![](figs/02-Shallow-and-Deep-Networks/deep-two-layer-3-3-diagram.png)


- 20 parameters
- (at least) 9 regions


![](figs/02-Shallow-and-Deep-Networks/shallow-six-hidden-units-diagram.png)


- 19 parameters
- Max 7 regions

[Why 7 regions?]{style="color: red;"}



## Composing networks in 2D

![](figs/02-Shallow-and-Deep-Networks/composing-networks-2d.png)

# Combining the two networks into one

Composing two shallow networks is a special case of a network with two hidden
layers. Substituting the first network into the second shows the structure, and
walking through the second layer's activations shows where the extra joints
come from.


## Combine two networks into one

*$\theta$: theta, $\phi$: phi*

Let's start with 2 networks:

**Network 1:** (input is $x$)

$$
\begin{aligned}
h_1 &= \mathrm{a}[\theta_{10} + \theta_{11} x] \\
h_2 &= \mathrm{a}[\theta_{20} + \theta_{21} x] \\
h_3 &= \mathrm{a}[\theta_{30} + \theta_{31} x]
\end{aligned}
$$

$$
y = \phi_0 + \phi_1 h_1 + \phi_2 h_2 + \phi_3 h_3
$$


**Network 2:** (input is $y$)

$$
\begin{aligned}
h'_1 &= \mathrm{a}[\theta'_{10} + \theta'_{11} y] \\
h'_2 &= \mathrm{a}[\theta'_{20} + \theta'_{21} y] \\
h'_3 &= \mathrm{a}[\theta'_{30} + \theta'_{31} y]
\end{aligned}
$$

$$
y' = \phi'_0 + \phi'_1 h'_1 + \phi'_2 h'_2 + \phi'_3 h'_3
$$


Substitute for $y$ to get hidden units of second network in terms of first:

$$
\begin{aligned}
h'_1 &= \mathrm{a}[\theta'_{10} + \theta'_{11} y] = \mathrm{a}[\theta'_{10} + \theta'_{11}\phi_0 + \theta'_{11}\phi_1 h_1 + \theta'_{11}\phi_2 h_2 + \theta'_{11}\phi_3 h_3] \\
h'_2 &= \mathrm{a}[\theta'_{20} + \theta'_{21} y] = \mathrm{a}[\theta'_{20} + \theta'_{21}\phi_0 + \theta'_{21}\phi_1 h_1 + \theta'_{21}\phi_2 h_2 + \theta'_{21}\phi_3 h_3] \\
h'_3 &= \mathrm{a}[\theta'_{30} + \theta'_{31} y] = \mathrm{a}[\theta'_{30} + \theta'_{31}\phi_0 + \theta'_{31}\phi_1 h_1 + \theta'_{31}\phi_2 h_2 + \theta'_{31}\phi_3 h_3]
\end{aligned}
$$



## Create new variables: $\psi$ (psi)

*$\theta$: theta, $\phi$: phi, $\psi$: psi*

Hidden units of 2^nd^ network in terms of hidden units of first network.


In each substituted expression above, group the constant terms
$\theta'_{\cdot 0} + \theta'_{\cdot 1}\phi_0$ and the coefficient
$\theta'_{\cdot 1}\phi_j$ of each $h_j$.


Collect and rename the variables for conciseness.

$$
\begin{aligned}
h'_1 &= \mathrm{a}[\psi_{10} + \psi_{11} h_1 + \psi_{12} h_2 + \psi_{13} h_3] \\
h'_2 &= \mathrm{a}[\psi_{20} + \psi_{21} h_1 + \psi_{22} h_2 + \psi_{23} h_3] \\
h'_3 &= \mathrm{a}[\psi_{30} + \psi_{31} h_1 + \psi_{32} h_2 + \psi_{33} h_3]
\end{aligned}
$$


## We get a two-layer network

$$
\begin{aligned}
h_1 &= \mathrm{a}[\theta_{10} + \theta_{11} x] \\
h_2 &= \mathrm{a}[\theta_{20} + \theta_{21} x] \\
h_3 &= \mathrm{a}[\theta_{30} + \theta_{31} x]
\end{aligned}
$$


$$
\begin{aligned}
h'_1 &= \mathrm{a}[\psi_{10} + \psi_{11} h_1 + \psi_{12} h_2 + \psi_{13} h_3] \\
h'_2 &= \mathrm{a}[\psi_{20} + \psi_{21} h_1 + \psi_{22} h_2 + \psi_{23} h_3] \\
h'_3 &= \mathrm{a}[\psi_{30} + \psi_{31} h_1 + \psi_{32} h_2 + \psi_{33} h_3]
\end{aligned}
$$


$$
y' = \phi'_0 + \phi'_1 h'_1 + \phi'_2 h'_2 + \phi'_3 h'_3
$$

![](figs/02-Shallow-and-Deep-Networks/two-layer-network-diagram.png)


## Two-layer network as one equation


Substituting the first-layer hidden units into the second layer, and those into
the output, gives the whole two-layer network as a single equation:


$$
\begin{aligned}
y' = \phi'_0 &+ \phi'_1 \mathrm{a}\left[\psi_{10} + \psi_{11}\mathrm{a}[\theta_{10} + \theta_{11}x] + \psi_{12}\mathrm{a}[\theta_{20} + \theta_{21}x] + \psi_{13}\mathrm{a}[\theta_{30} + \theta_{31}x]\right] \\
&+ \phi'_2 \mathrm{a}[\psi_{20} + \psi_{21}\mathrm{a}[\theta_{10} + \theta_{11}x] + \psi_{22}\mathrm{a}[\theta_{20} + \theta_{21}x] + \psi_{23}\mathrm{a}[\theta_{30} + \theta_{31}x]] \\
&+ \phi'_3 \mathrm{a}[\psi_{30} + \psi_{31}\mathrm{a}[\theta_{10} + \theta_{11}x] + \psi_{32}\mathrm{a}[\theta_{20} + \theta_{21}x] + \psi_{33}\mathrm{a}[\theta_{30} + \theta_{31}x]]
\end{aligned}
$$

## 2nd layer pre-activations = a shallow network with three outputs


Consider the pre-activations at the second hidden units. Up to this point the
network is a shallow network with three outputs, and like the two-output
example earlier, all three pre-activations have their joints at the same
places.

![The network up to the pre-activations of the second hidden layer.](figs/02-Shallow-and-Deep-Networks/first-layer-three-outputs-highlighted.png)

![The three second-layer pre-activations as functions of $x$: three outputs of a shallow network, with joints at the same two positions.](figs/02-Shallow-and-Deep-Networks/second-layer-preactivations.png)


## 2nd layer: activate, weight, sum


The second layer repeats the four-step construction on the pre-activations: the
ReLU clips each one where it is negative, which adds new joints; the clipped
functions are weighted by $\phi'_1, \phi'_2, \phi'_3$; and the weighted
functions are summed with the offset $\phi'_0$.

![Second layer, steps 2 and 4. Top row: the activations $h'_1, h'_2, h'_3$ after the ReLU, each with new joints where its pre-activation crossed zero. Bottom: the weighted activations summed and offset to give $y'$.](figs/02-Shallow-and-Deep-Networks/second-layer-summed-output.png)


2^nd^ analogy: create new functions which are clipped and recombined. Each
layer can add joints inside every existing region.


## Try it: the second layer

Drag the handles in the top row (the three pre-activations share their joints)
and move the $\phi'$ sliders.




[Interactive version of UDL Figure 4.5, ported from the UDL interactive figures (MIT).]{style="font-size: 0.7em;"}

Top row: the layer-2 pre-activations, three outputs of a shallow network, so
their joints line up, so a joint dragged in one panel moves in all three.
Middle row: after the ReLU, which adds a joint wherever a
pre-activation crosses zero. Bottom row: weighted by $\phi'_1, \phi'_2,
\phi'_3$; their sum plus $\phi'_0$ is the output on the right.


# Hyperparameters

The number of layers and the number of hidden units per layer are design
choices made before training.


## Hyperparameters

- $K$ layers = [depth of network]{style="color: #D2805F;"}
- $D_k$ hidden units in layer $k$ = [width of network]{style="color: #D2805F;"}

- These are called [hyperparameters]{style="color: #D2805F;"}: chosen before training the network, not learned by gradient descent
- Can try retraining with different hyperparameters: [hyperparameter optimization]{style="color: #D2805F;"} or [hyperparameter search]{style="color: #D2805F;"}
  - This can be either manual or automated (e.g. [Hyperparameter Tuning with Ray Tune](https://pytorch.org/tutorials/beginner/hyperparameter_tuning_tutorial.html))


# Matrix notation and the general case

Writing out every hidden unit does not scale. Three notation changes turn the
two-layer equations into a compact form that generalizes to any depth, and that
is also how the network is implemented: each layer is a matrix multiply, a
bias add and an elementwise activation.


## Notation change #1: vectors and matrices

$$
\begin{aligned}
h_1 &= \mathrm{a}[\theta_{10} + \theta_{11} x] \\
h_2 &= \mathrm{a}[\theta_{20} + \theta_{21} x] \\
h_3 &= \mathrm{a}[\theta_{30} + \theta_{31} x]
\end{aligned}
$$


[*Vector notation*]{style="color: #7030A0;"}

$$
\begin{bmatrix} h_1 \\ h_2 \\ h_3 \end{bmatrix}
= \mathbf{a}\left[\begin{bmatrix} \theta_{10} \\ \theta_{20} \\ \theta_{30} \end{bmatrix}
+ \begin{bmatrix} \theta_{11} \\ \theta_{21} \\ \theta_{31} \end{bmatrix} x\right]
$$


$$
\begin{aligned}
h'_1 &= \mathrm{a}[\psi_{10} + \psi_{11} h_1 + \psi_{12} h_2 + \psi_{13} h_3] \\
h'_2 &= \mathrm{a}[\psi_{20} + \psi_{21} h_1 + \psi_{22} h_2 + \psi_{23} h_3] \\
h'_3 &= \mathrm{a}[\psi_{30} + \psi_{31} h_1 + \psi_{32} h_2 + \psi_{33} h_3]
\end{aligned}
$$


$$
\begin{bmatrix} h'_1 \\ h'_2 \\ h'_3 \end{bmatrix}
= \mathbf{a}\left[\begin{bmatrix} \psi_{10} \\ \psi_{20} \\ \psi_{30} \end{bmatrix}
+ \begin{bmatrix} \psi_{11} & \psi_{12} & \psi_{13} \\ \psi_{21} & \psi_{22} & \psi_{23} \\ \psi_{31} & \psi_{32} & \psi_{33} \end{bmatrix}
\begin{bmatrix} h_1 \\ h_2 \\ h_3 \end{bmatrix}\right]
$$

[*Vector & matrix notation*]{style="color: #7030A0;"}


$$
y' = \phi'_0 + \phi'_1 h'_1 + \phi'_2 h'_2 + \phi'_3 h'_3
$$


$$
y' = \phi'_0 + \begin{bmatrix} \phi'_1 & \phi'_2 & \phi'_3 \end{bmatrix}
\begin{bmatrix} h'_1 \\ h'_2 \\ h'_3 \end{bmatrix}
$$


## Notation change #2: bold symbols

[Notation reminder]{style="text-decoration: underline;"}<br>
$x, \psi$ : normal lower case -- scalar<br>
$\mathbf{x}, \boldsymbol{\psi}$ : bold face lower case -- vector<br>
$\mathbf{X}, \boldsymbol{\Psi}$ : bold face upper case -- matrix



Writing the vectors and matrices from notation change #1 in bold:

$$
\begin{aligned}
\mathbf{h} &= \mathbf{a}[\boldsymbol{\theta}_0 + \boldsymbol{\theta}_1 x] \\
\mathbf{h}' &= \mathbf{a}[\boldsymbol{\psi}_0 + \boldsymbol{\Psi}\mathbf{h}] \\
y' &= \phi'_0 + \boldsymbol{\phi}'^{T}\mathbf{h}'
\end{aligned}
$$


## Notation change #3: one symbol per layer

[$\beta$ : beta, $\Omega$ : Omega]{style="font-size: 0.7em;"}

$$
\begin{aligned}
\mathbf{h} &= \mathbf{a}\left[\boldsymbol{\theta}_0 + \boldsymbol{\theta}_1 x\right]
&& \longrightarrow &
\mathbf{h}_1 &= \mathbf{a}[\boldsymbol{\beta}_0 + \boldsymbol{\Omega}_0 \mathbf{x}] \\
\mathbf{h}' &= \mathbf{a}\left[\boldsymbol{\psi}_0 + \boldsymbol{\Psi}\mathbf{h}\right]
&& \longrightarrow &
\mathbf{h}_2 &= \mathbf{a}[\boldsymbol{\beta}_1 + \boldsymbol{\Omega}_1 \mathbf{h}_1] \\
y' &= \phi'_0 + \boldsymbol{\phi}'^{T}\mathbf{h}'
&& \longrightarrow &
\mathbf{y} &= \boldsymbol{\beta}_2 + \boldsymbol{\Omega}_2 \mathbf{h}_2
\end{aligned}
$$

- $\boldsymbol{\beta}_k$: [bias vector]{style="color: #D2805F;"}
- $\boldsymbol{\Omega}_k$: [weight matrix]{style="color: #D2805F;"}
- The input $\mathbf{x}$ and output $\mathbf{y}$ are now vectors too: $D_i$
  inputs and $D_o$ outputs for free


## General equations for a deep network

$$
\begin{aligned}
\mathbf{h}_1 &= \mathbf{a}[\boldsymbol{\beta}_0 + \boldsymbol{\Omega}_0 \mathbf{x}] \\
\mathbf{h}_2 &= \mathbf{a}[\boldsymbol{\beta}_1 + \boldsymbol{\Omega}_1 \mathbf{h}_1] \\
\mathbf{h}_3 &= \mathbf{a}[\boldsymbol{\beta}_2 + \boldsymbol{\Omega}_2 \mathbf{h}_2] \\
&\;\;\vdots \\
\mathbf{h}_K &= \mathbf{a}[\boldsymbol{\beta}_{K-1} + \boldsymbol{\Omega}_{K-1} \mathbf{h}_{K-1}] \\
\mathbf{y} &= \boldsymbol{\beta}_K + \boldsymbol{\Omega}_K \mathbf{h}_K,
\end{aligned}
$$

$$
\mathbf{y} = \boldsymbol{\beta}_K + \boldsymbol{\Omega}_K \mathbf{a}\left[\boldsymbol{\beta}_{K-1} + \boldsymbol{\Omega}_{K-1}\mathbf{a}\left[\ldots \boldsymbol{\beta}_2 + \boldsymbol{\Omega}_2 \mathbf{a}\left[\boldsymbol{\beta}_1 + \boldsymbol{\Omega}_1 \mathbf{a}\left[\boldsymbol{\beta}_0 + \boldsymbol{\Omega}_0 \mathbf{x}\right]\right]\ldots\right]\right]
$$

If layer $k$ has $D_k$ units, then $\boldsymbol{\Omega}_k \in \mathbb{R}^{D_{k+1} \times D_k}$
and $\boldsymbol{\beta}_k \in \mathbb{R}^{D_{k+1}}$ (with $D_0 = D_i$ and $D_{K+1} = D_o$).


## Example: counting parameters

![](figs/02-Shallow-and-Deep-Networks/deep-network-example.png), Omega_1 in R^{2x4}, Omega_2 in R^{3x2}, Omega_3 in R^{2x3}." width="100%"}

**Question:** how many parameters does this network have?



A parameter counter for a fully connected network given its widths
$[D_i, D_1, \ldots, D_K, D_o]$: layer $k$ contributes $D_{k+1} \times D_k$
weights and $D_{k+1}$ biases. It reproduces this slide's answer and the counts
from earlier in the lecture.

In [ ]:
def count_params(widths):
    """widths = [D_i, D_1, ..., D_K, D_o]."""
    return sum(d_out * (d_in + 1) for d_in, d_out in zip(widths[:-1], widths[1:]))

examples = {
    "shallow 3-3-2":          [3, 3, 2],
    "shallow 1-6-1":          [1, 6, 1],
    "two-layer 1-3-3-1":      [1, 3, 3, 1],
    "deep 3-4-2-3-2 (above)": [3, 4, 2, 3, 2],
}
for name, widths in examples.items():
    print(f"{name:<24} {count_params(widths):>3} parameters")

# Shallow vs. deep networks

Both shallow and deep networks can approximate any function, so why are the
best results produced by deep networks? The slides below go through five
possible reasons. The honest answer is that the second (regions per parameter)
and third (depth efficiency) are proven properties of the function class, while
whether real-world functions exploit them is still unknown.


## Shallow vs. deep networks

The best results are created by deep networks with many layers.

- 50-1000 layers for most applications
- Best results in
  - Computer vision
  - Natural language processing
  - Graph neural networks
  - Generative models
  - Reinforcement learning


All use deep networks. But why?


**1. Ability to approximate different functions?** Both obey the universal
approximation theorem. One layer is enough, and a deep network can arrange for
the other layers to compute the identity function.


## 2. Number of linear regions per parameter

![Figure 4.7: Prince, *Understanding Deep Learning*.](figs/02-Shallow-and-Deep-Networks/linear-regions-1d-10d.png)

**a)** Each small dot is an additional hidden unit per layer.

Large dot: 5 layers, 10 hidden units per layer, 471 parameters, 161,051 linear
regions


**b)** Each small dot is an additional 10 hidden units per layer.

Large dot: 5 layers, 50 hidden units per layer, 10,801 parameters, $>10^{40}$
linear regions



For $K$ hidden layers of $D \geq D_i$ units, Montúfar et al. (2014) construct
networks with

$$
\left(\frac{D}{D_i} + 1\right)^{D_i (K - 1)} \sum_{j=0}^{D_i} \binom{D}{j}
$$

linear regions (UDL chapter 4 notes). This is the curve plotted in Figure 4.7;
the calculator below reproduces both large dots and compares each with a
shallow network of about the same parameter count.

In [ ]:
def regions_deep(D_i, D, K):
    """Regions achievable with K hidden layers of D >= D_i units (Montufar et al., 2014)."""
    return (D // D_i + 1) ** (D_i * (K - 1)) * max_regions_shallow(D_i, D)

def fmt(n):
    return f"{n:,}" if n < 10**9 else f"about 10^{log10(n):.1f}"

for D_i, D, K in [(1, 10, 5), (10, 50, 5)]:
    N = regions_deep(D_i, D, K)
    P = count_params([D_i] + [D] * K + [1])
    D_shallow = (P - 1) // (D_i + 2)            # one hidden layer, about the same parameter count
    P_shallow = count_params([D_i, D_shallow, 1])
    print(f"D_i = {D_i:>2}, deep    K = {K}, D = {D:>3}: {P:>6,} parameters, {fmt(N)} regions")
    print(f"D_i = {D_i:>2}, shallow K = 1, D = {D_shallow:>3}: {P_shallow:>6,} parameters, "
          f"{fmt(max_regions_shallow(D_i, D_shallow))} regions")

## 2 (cont.) and 3: regions per parameter, depth efficiency

**2. Number of linear regions per parameter**

- Deep networks create many more regions per parameter
- But there are dependencies between them
  - Think of the folding example: the regions repeat the same function
  - Perhaps similar symmetries in real-world functions? Unknown

**3. Depth efficiency**

- There are some functions that require a shallow network with exponentially
  more hidden units than a deep network to achieve an equivalent approximation
- This is known as the [depth efficiency]{style="color: #D2805F;"} of deep networks
- But do the real-world functions we want to approximate have this property?
  Unknown.


## 4 and 5: structure, fitting and generalization

**4. Large structured networks**

- Think about images as input: might be 1M pixels, so fully connected networks are not practical
- Answer is to have weights that only operate locally, and share across the image
- This leads to [convolutional networks]{style="color: #D2805F;"}
- Gradually integrate information from across the image: needs multiple layers

**5. Fitting and generalization**

- Fitting of deep models seems to be easier up to about 20 layers
- Then needs various tricks to train deeper networks, so (in vanilla form),
  fitting becomes harder
- Generalization is good in deep networks. Why? Still an open question.


## 5 (cont.): fitting deep models is also faster

**Figure 20.2** MNIST-1D training. Four fully connected networks were fit to
4000 MNIST-1D examples with random labels using full batch gradient descent, He
initialization, no momentum or regularization, and learning rate 0.0025. Models
with 1,2,3,4 layers had 298, 100, 75, and 63 hidden units per layer and 15208,
15210, 15235, and 15319 parameters, respectively. All models train successfully,
but deeper models require fewer epochs.


![Figure: Prince, *Understanding Deep Learning*, Fig. 20.2.](figs/02-Shallow-and-Deep-Networks/mnist1d-train-error-by-depth.png)


The book caption says 15139 parameters for the 4-layer model; recomputed for a
40-63-63-63-63-10 network it is $40 \cdot 63 + 63 + 3 \cdot (63 \cdot 64) + 63 \cdot 10 + 10 = 15{,}319$.

In [ ]:
for depth, width in [(1, 298), (2, 100), (3, 75), (4, 63)]:
    print(f"{depth} hidden layer(s) of {width:>3}: "
          f"{count_params([40] + [width] * depth + [10]):,} parameters")

# Summary

Try it yourself: at [playground.tensorflow.org](https://playground.tensorflow.org/)
set up 2 inputs, 3 hidden units and 1 output with ReLU, and check whether the
decision boundary shows the number of regions you expect, then add a second
layer and watch the regions multiply.


## Summary

- A shallow network is a weighted sum of hidden units, each a ReLU of a linear
  function of the input: $h_d = \mathrm{a}[\theta_{d0} + \sum_i \theta_{di} x_i]$,
  $y_j = \phi_{j0} + \sum_d \phi_{jd} h_d$
- With ReLU, the output is piecewise linear: one joint (hyperplane) per hidden
  unit, linear inside each region, and the activation pattern is fixed within a region
- Universal approximation: with enough hidden units a shallow network
  approximates any continuous function; it does not say how many, or that
  training will find them
- $D$ hidden units in $D_i$ dimensions make at most $\sum_{j=0}^{D_i} \binom{D}{j}$
  regions (Zaslavsky), between $2^{D_i}$ and $2^D$
- Composing shallow networks replicates one function inside each region of the
  other (folding), so the regions multiply
- A deep network is a chain of layers $\mathbf{h}_{k+1} = \mathbf{a}[\boldsymbol{\beta}_k + \boldsymbol{\Omega}_k \mathbf{h}_k]$;
  depth $K$ and widths $D_k$ are hyperparameters, and the parameter count is
  $\sum_k D_{k+1}(D_k + 1)$
- Deep networks give far more linear regions per parameter than shallow ones
  (depth efficiency), but whether real-world functions exploit this is unknown

**Where are we going?** We have defined families of very flexible networks
that map multiple inputs to multiple outputs. Now we need to train them: how
to choose loss functions for different types of targets (next lecture; read
UDL chapter 5), how to find minima of the loss function, and how to do this
efficiently with deep networks. Then, how do we evaluate them?
